# **`Agent_01(Triage Agent)`**

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('EGROQ_API_KEY')
client = Groq(api_key=api_key)
print("Connected!")

Connected!


In [ ]:
sample_email_subject="URGENT: Your account has been locked.Verify Now!"
sample_email_body="Dear customer,we have detected unusual activity on your account.Please verify your identity immediately by clicking the link below.Failure to do so within 24 hours may result in permanent aaccount suspension."

In [ ]:
def triage_agent(subject,body):
  prompt=f"""
  You are an email security triage agent.Analyze the following email and classify it as one of:Important,Spam, or Suspicious.

  Look for these patterns:
  -Urgency-based language(e.g,"act now", "24 hours","immediately")
  -Requests for sensitive information or account verification
  -Threats of account suspension or loss
  -Generic greetings instead of personalized ones.


Subject:{subject}
Body:{body}

Respond ONLY in this JSON format:
{{
"classification":"Important,Spam,or Suspicious",
"reasons":["reason1","reason2"],
"recommendation":"what the user should do"
}}
"""
  response=client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":prompt}]
  )
  return response.choices[0].message.content

#Test
result=triage_agent(sample_email_subject,sample_email_body)
print(result)
#

{
  "classification": "Spam",
  "reasons": [
    "Urgency-based language with demands to act immediately",
    "Request for verification of identity through suspicious link",
    "Threat of permanent account suspension without a clear explanation"
  ],
  "recommendation": "Do not click the provided link and contact the legitimate account support directly to verify your account status"
}


# **`TESTING (Important Email and Spam Email)`**

In [ ]:
# Important email test
important_subject = "Meeting rescheduled to 3 PM tomorrow"
important_body = "Hi team, our project sync meeting has been moved from 10 AM to 3 PM tomorrow due to a scheduling conflict. Please update your calendars. Thanks, Sarah"

result_important = triage_agent(important_subject, important_body)
print(result_important)

{
"classification": "Important",
"reasons": ["Urgency-based language was not present but the request was personalized", "The email contained a specific and clear invitation with no suspicious language"],
"recommendation": "Update calendar with the new meeting time as instructed"
}


In [ ]:
# Spam email test
spam_subject = "You've won a $1000 gift card!"
spam_body = "Congratulations! You have been randomly selected to receive a free $1000 Amazon gift card. Click here to claim your prize now before it expires!"

result_spam = triage_agent(spam_subject, spam_body)
print(result_spam)

{
"classification":"Spam",
"reasons":["Urgency-based language is used to create a sense of FOMO","Generic greeting instead of personalized one"],
"recommendation":"Delete the email immediately and do not interact with any links or click on the claim prize button."
}


##**` Agent_02(Phishing intelligence & Link Verification Agent)`**

In [ ]:
import re

def extract_urls(email_body):
    url_pattern = r'https?://[^\s]+'
    urls = re.findall(url_pattern, email_body)
    return urls

In [ ]:
def link_verification_agent(url, brand_context=""):
    prompt = f"""
    You are a phishing intelligence agent. Analyze this URL and determine
    if it is genuine or a phishing/scam attempt.

    Check for:
    - Brand impersonation (e.g., "amaz0n" instead of "amazon", extra words/dashes)
    - Suspicious domain patterns (unusual TLDs like .xyz, .top, misspellings)
    - Whether the URL matches the claimed sender/brand in the email context

    URL: {url}
    Email context: {brand_context}

    Respond ONLY in this JSON format:
    {{
        "verdict": "genuine/phishing",
        "trust_score": 0-100,
        "risk_level": "LOW/MEDIUM/HIGH",
        "reasons": ["reason1", "reason2"],
        "recommendation": "what the user should do"
    }}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# **`Testing Agent_02`**

In [ ]:
# Test 1: Phishing URL
result1 = link_verification_agent(
    "http://amaz0n-offers.xyz/track",
    "Amazon Order Confirmation"
)
print("PHISHING TEST:")
print(result1)
print()

# Test 2: Genuine URL
result2 = link_verification_agent(
    "https://www.amazon.com/orders/track",
    "Amazon Order Confirmation"
)
print("GENUINE TEST:")
print(result2)

PHISHING TEST:
{
    "verdict": "phishing",
    "trust_score": 25,
    "risk_level": "HIGH",
    "reasons": [
        "Brand impersonation (amaz0n instead of Amazon)",
        "Suspicious domain TLD (.xyz instead of .com)",
        "Non-standard subdomain (amaz0n-offers)"
    ],
    "recommendation": "Do not click on the link and report it as phishing to Amazon's support team."
}

GENUINE TEST:
{
    "verdict": "genuine",
    "trust_score": 90,
    "risk_level": "LOW",
    "reasons": [
        "The URL matches the claimed sender/brand in the email context (Amazon Order Confirmation).",
        "The URL does not display any brand impersonation (e.g., misspellings or extra words/dashes).",
        "The domain name (amazon.com) has a trusted TLD (.com) and is not suspicious."
    ],
    "recommendation": "You can safely click on the link to track your Amazon order."
}


# **`Agent-to-Agent Communication: Full Pipeline (Agent 1 → Agent 2)`**

This section demonstrates the collaborative workflow between the Triage
Agent and the Link Verification Agent. Agent 1 classifies the incoming
email and passes the extracted context to Agent 2, which verifies any
embedded URLs. This structured hand-off between agents represents the
system's agent-to-agent communication protocols.


---




In [ ]:
def process_email(subject, body):
    # Agent 1: Triage
    triage_result = triage_agent(subject, body)
    print("=== AGENT 1 (Triage) OUTPUT ===")
    print(triage_result)

    # Extract URLs
    urls = extract_urls(body)

    # Agent 2: Only runs if there are links
    if urls:
        print("\n=== AGENT 2 (Link Verification) OUTPUT ===")
        for url in urls:
            link_result = link_verification_agent(url, subject)
            print(link_result)
    else:
        print("\nNo links found in this email. Skipping link verification.")


In [ ]:
sample_url_email_subject = "PayPal Security Alert"
sample_url_email_body = (
    "We detected unusual activity on your account. "
    "Verify your identity immediately at: http://paypa1-secure-login.xyz/verify"
)

label = "phishing"

In [ ]:
# Test the full pipeline
process_email(sample_url_email_subject, sample_url_email_body)

=== AGENT 1 (Triage) OUTPUT ===
{
"classification": "Suspicious",
"reasons": [
    "Generic greeting, which is uncommon in legitimate PayPal alerts",
    "Urgency-based language ('Verify your identity immediately') could be an attempt to rush the user into a potentially insecure action",
    "Use of a generic URL that does not match the verified PayPal login page ('http://paypa1-secure-login.xyz/verify')",
    "Request for sensitive information (identity verification) via an unverified source"
],
"recommendation": "Do not click on the link and instead log in to your PayPal account through the official website to verify your information."
}

=== AGENT 2 (Link Verification) OUTPUT ===
{
    "verdict": "phishing",
    "trust_score": 20,
    "risk_level": "HIGH",
    "reasons": [
        "Brand impersonation: 'paypa1' instead of 'PayPal', indicating an attempt to mimic the brand.",
        "Suspicious domain pattern: '.xyz' TLD, which is commonly associated with phishing attempts.",
      

In [ ]:
def reflect_on_verdict(initial_result, url):
    prompt = f"""
    Review this phishing analysis and verify if the verdict is accurate.
    Confirm or correct the assessment if needed.

    Initial Analysis: {initial_result}
    URL: {url}

    Respond in the same JSON format as before.
    """
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test
reflection_result = reflect_on_verdict(result1, "http://amaz0n-offers.xyz/track")
print(reflection_result)

**Verified Analysis:**

{
    "verdict": "phishing",
    "trust_score": 70,  // Decreased the trust score based on corrected reasoning
    "risk_level": "MEDIUM",  // Adjusted the risk level based on corrected reasoning
    "reasons": [
        "Brand impersonation (amaz0n instead of Amazon)"  // Verified
        "Suspicious domain TLD (.xyz instead of .com)"  // Not necessarily correct: .xyz is a valid TLD, although not typical for Amazon.
        "Non-standard subdomain (amaz0n-offers)"  // Verified
        "Missing standard HTTP security certificate (HTTPS)"  // Corrected reason to include missing HTTPS security
    ],
    "recommendation": "Use caution when interacting with this URL, but further analysis may be needed as the risk level has decreased. Do not click on the link without verifying Amazon's authenticity."
}

Note:
- The trust score has decreased as only two out of the three initial reasons were verified as correct.
- The risk level has decreased as missing a standard HTT

In [ ]:
!pip install openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

openrouter_key = userdata.get('OPENROUTER_API_KEY')
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key
)
print("OpenRouter Connected!")

OpenRouter Connected!


In [ ]:
def reflect_on_verdict(initial_result, url):
    prompt = f"""
    Review this phishing analysis and verify if the verdict is accurate.
    Confirm or correct the assessment if needed.

    Initial Analysis: {initial_result}
    URL: {url}

    Respond in the same JSON format as before.
    """
    response = openrouter_client.chat.completions.create(
        model="meta-llama/llama-3.2-3b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
reflection_result = reflect_on_verdict(result1, "http://amaz0n-offers.xyz/track")
print(reflection_result)

Here is the review of the phishing analysis:

{
  "verdict": "phishing",
  "trust_score": 25,
  "risk_level": "HIGH",
  "reasons": [
    "Brand impersonation (amaz0n instead of Amazon)",
    "Suspicious domain TLD (.xyz instead of .com)",
    "Non-standard subdomain (amaz0n-offers)"
  ],
  "recommendation": "Do not click on the link and report it as phishing to Amazon's support team."
}

**Verdict:** Accurate

**Recommendation:** Accurate

The analysis correctly identified the phishing attempt as:
- "Brand impersonation" due to the misspelling of "Amazon" as "amaz0n".
- The domain extension ".xyz" is indeed not trusted compared to the more traditional and secure ".com".
- The use of a non-standard subdomain "amaz0n-offers" further increases the risk.

To mitigate the issue, the recommendation to not click on the link and report it to Amazon's support team is correct, as it can help prevent potential scams and protect user's credentials and sensitive information.


# `Reflection + OpenRouter`

In [ ]:
!pip install openai

from openai import OpenAI

openrouter_key = userdata.get("OPENROUTER_API_KEY")
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key
)
print("OpenRouter Connected!")

OpenRouter Connected!


In [ ]:
def reflect_on_verdict(initial_result, url):
    prompt = f"""
    Review this phishing analysis and verify if the verdict is accurate.
    Confirm or correct the assessment if needed.

    Initial Analysis: {initial_result}
    URL: {url}

    Respond in the same JSON format as before.
    """
    response = openrouter_client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
reflection_result = reflect_on_verdict(result1, "http://amaz0n-offers.xyz/track")
print(reflection_result)

Here is the verified review in JSON format:

{
  "verdict": "phishing",
  "trust_score": 25,
  "risk_level": "HIGH",
  "reasons": [
    "Domain ownership uncertain (amaz0n-offers.xyz is not an official Amazon subdomain)",
    "Suspicious domain TLD (.xyz is not a standard TLD for Amazon or official Amazon sellers)",
    "Non-standard TLD and subdomain used",
    "Brand impersonation (amaz0n instead of Amazon)"
  ],
  "recommendation": "Do not click on the link and report it as phishing to Amazon's support team."
}

I verified the original assessment, and the verdict is still accurate as the website is indeed a phishing attempt. The reasons for this conclusion include:

- The domain appears to be owned by someone unclear (amaz0n-offers.xyz), as they are not an official Amazon subdomain.
- The use of the .xyz top-level domain is not standard for Amazon or official Amazon sellers.
- The link includes a non-standard subdomain (amaz0n) instead of the official spelling ("Amazon").

The recom